# 09 — Local RL Debug Run

Run a fast end-to-end PPO training loop locally and inspect every artifact it produces.

**What this notebook does:**
1. Configure key training params in one cell.
2. Launch `run_training.py` via shell and capture its output.
3. Inspect the resulting `manifest.json`, JSONL logs, episode JSONs, and checkpoint.

**Runtime:** ~30–90 seconds depending on `NUM_TIMESTEPS` and LLM latency.  
**No GPU required** — runs on CPU by default (`device: cpu` in `ppo_debug.yaml`).

---

### Relationship to the test suite

The unit / system tests mock the LLM and the environment — they verify logic, not artifacts.  
This notebook exercises the **real stack** end-to-end and lets you read the outputs.
Use it to:
- Confirm a new compressor wires up correctly before a Colab run.
- Verify JSONL schema after changing `RLRunLogger` or `RewardFunction`.
- Spot-check episode trajectories for agent behaviour regressions.

## 1. Configure the run

Edit the variables below. Everything else is derived from `ppo_debug.yaml`.

| Variable | What it controls | Guideline |
|---|---|---|
| `NUM_TIMESTEPS` | Total env steps | 8 = 2 PPO updates; 32 = ~8 updates |
| `N_ENVS` | Parallel simulator instances | 1 (single-threaded, easiest to debug) |
| `N_STEPS` | Steps per env per rollout | Must be ≥ `NUM_TIMESTEPS / N_ENVS` to trigger at least one update |
| `MAX_AGENT_STEPS` | ReAct steps per episode (= LLM calls) | 2 is minimal; 4 gives richer trajectories |
| `COMPRESSOR` | Which compressor to use | `identity` (no API), `llm` (needs key), `dummy` (random) |
| `RUN_NAME` | Tag shown in Streamlit + manifest | Any string, no spaces |

In [ ]:
# ── Tune these ────────────────────────────────────────────────────────────────
NUM_TIMESTEPS   = 8      # 8 = ~2 PPO updates (fastest possible)
N_ENVS          = 1
N_STEPS         = 4      # rollout buffer fills after N_STEPS * N_ENVS env steps
MAX_AGENT_STEPS = 2      # ReAct steps per episode (each = 1 LLM call)
COMPRESSOR      = 'identity'   # identity | llm | dummy | transformer
RUN_NAME        = 'local_debug'

# ── Derived (don't edit) ──────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT   = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_DIR  = REPO_ROOT / 'outputs'
TRAINING_DIR = OUTPUT_DIR / 'training'
EPISODES_DIR = OUTPUT_DIR / 'episodes'
sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Repo root  : {REPO_ROOT}')
print(f'Compressor : {COMPRESSOR}')
print(f'Steps      : {NUM_TIMESTEPS} total, {N_STEPS}/rollout, {N_ENVS} envs')
print(f'Run name   : {RUN_NAME}')

## 2. Launch training

The command below uses `ppo_debug.yaml` as the base config and overrides the variables you set above.  
Run the cell and watch the output — it shows per-episode reward and PPO diagnostics.

In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'run_training.py'),
    f'compressor={COMPRESSOR}',
    'training=ppo_debug',
    f'training.num_timesteps={NUM_TIMESTEPS}',
    f'training.n_envs={N_ENVS}',
    f'training.ppo.n_steps={N_STEPS}',
    f'training.ppo.batch_size={N_STEPS * N_ENVS}',
    f'agent.max_steps={MAX_AGENT_STEPS}',
    f'training.run_name={RUN_NAME}',
    'training.episode_save_freq=1',   # save every episode so we can inspect them
    'training.checkpoint_every_n_steps=8',
    'hydra.run.dir=outputs/hydra_runs/${now:%Y%m%d_%H%M%S}',
]

print('Command:', ' '.join(cmd[1:]))
print('─' * 60)

t0 = time.time()
result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=False)
elapsed = time.time() - t0

print('─' * 60)
if result.returncode == 0:
    print(f'Training complete in {elapsed:.1f}s')
else:
    print(f'Training FAILED (exit code {result.returncode}) after {elapsed:.1f}s')

## 3. Inspect the run manifest

`manifest.json` captures the full resolved config at the moment training started.  
It is the single source of truth for what a run did.

In [ ]:
from optimized_llm_planning_memory.training.run_manifest import list_manifests

manifests = list_manifests(TRAINING_DIR)
if not manifests:
    print('No manifests found. Did the training run complete?')
else:
    m = manifests[0]   # most recent run
    print(f'run_id          : {m.run_id}')
    print(f'run_name        : {m.run_name}')
    print(f'compressor_type : {m.compressor_type}')
    print(f'agent_mode      : {m.agent_mode}')
    print(f'n_train_requests: {m.n_train_requests}')
    print(f'num_timesteps   : {m.num_timesteps}')
    print(f'n_envs          : {m.n_envs}')
    print(f'created_at      : {m.created_at}')
    print()
    print('PPO hyperparams:')
    for k, v in sorted(m.ppo_hyperparams.items()):
        print(f'  {k:<25}: {v}')
    print()
    print('Reward weights:')
    for k, v in sorted(m.reward_weights.items()):
        print(f'  {k:<25}: {v}')
    
    RUN_ID = m.run_id   # carry forward to subsequent cells

## 4. Episode metrics (JSONL)

One record per completed episode. Shows reward breakdown, tool stats, and compression count.
This is the lean training log — readable without TensorBoard.

In [ ]:
import pandas as pd
from optimized_llm_planning_memory.training.run_logger import load_episode_metrics

ep_records = load_episode_metrics(RUN_ID, TRAINING_DIR)
if not ep_records:
    print('No episode metrics found.')
else:
    ep_df = pd.DataFrame([r.model_dump() for r in ep_records])
    print(f'{len(ep_df)} episode records')
    pd.set_option('display.max_columns', None)
    pd.set_option('display.float_format', '{:.3f}'.format)
    display(ep_df)

In [ ]:
import matplotlib.pyplot as plt

if len(ep_df) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    for ax, col, label in [
        (axes[0], 'total_reward',          'Total reward'),
        (axes[1], 'hard_constraint_score', 'Hard constraint score'),
        (axes[2], 'tool_efficiency_score', 'Tool efficiency'),
    ]:
        if col in ep_df:
            ax.plot(ep_df[col].values, marker='o', markersize=4)
            ax.set_title(label)
            ax.set_xlabel('Episode')
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Only 1 episode — not enough for a meaningful plot.')

## 5. PPO update diagnostics (JSONL)

One record per PPO update cycle. Shows convergence indicators.

| Metric | Healthy range | Red flag |
|---|---|---|
| `approx_kl` | < 0.02 | > 0.05 (LR too high) |
| `clip_fraction` | 0.1–0.2 | > 0.3 |
| `explained_variance` | trending → 1.0 | stuck near 0 |
| `policy_loss` | decreasing | exploding (nan/inf) |

In [ ]:
from optimized_llm_planning_memory.training.run_logger import load_ppo_metrics

ppo_records = load_ppo_metrics(RUN_ID, TRAINING_DIR)
if not ppo_records:
    print('No PPO update records found (not enough timesteps for an update?)')
else:
    ppo_df = pd.DataFrame([r.model_dump() for r in ppo_records])
    print(f'{len(ppo_df)} PPO update records')
    display(ppo_df.round(5))

## 6. Inspect a saved episode log

When `episode_save_freq=1`, every episode is saved as a full `EpisodeLog` JSON.  
This lets you read the exact ReAct trajectory, tool calls, compressed states, and reward breakdown.

In [ ]:
import json

ep_files = sorted(EPISODES_DIR.glob('ep_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
print(f'{len(ep_files)} episode JSON files found')
for f in ep_files[:5]:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

In [ ]:
# Pick episode to inspect (0 = most recent)
EPISODE_INDEX = 0

if ep_files:
    ep_path = ep_files[EPISODE_INDEX]
    raw = json.loads(ep_path.read_text(encoding='utf-8'))

    print(f'Episode: {raw["episode_id"]}')
    print(f'Request: {raw["request_id"]}')
    print(f'Agent mode: {raw["agent_mode"]}')
    print(f'Total steps: {raw["total_steps"]}')
    print(f'Success: {raw["success"]}')
    print(f'Termination reason: {raw.get("termination_reason", "N/A")}')
    print()

    # Reward breakdown
    rc = raw.get('reward_components', {})
    print('Reward components:')
    for k, v in rc.items():
        if isinstance(v, (int, float)):
            print(f'  {k:<35} {v:.4f}')

    # Tool call stats
    stats = raw.get('tool_call_stats', {})
    print(f'\nTool stats: total={stats.get("total_calls", 0)}, '
          f'success={stats.get("successful_calls", 0)}, '
          f'failed={stats.get("failed_calls", 0)}')
else:
    print('No episode files found. Check that episode_save_freq > 0 was set.')

In [ ]:
# Print the ReAct trajectory steps
if ep_files:
    traj = raw.get('trajectory', {})
    steps = traj.get('steps', [])
    print(f'{len(steps)} trajectory steps\n')
    for step in steps:
        print(f'Step {step["step_index"]}')
        print(f'  Thought  : {step["thought"][:120]}')
        if step.get('action'):
            act = step['action']
            print(f'  Action   : {act.get("tool_name", "?")}({act.get("arguments", {})})')
        if step.get('observation'):
            obs = step['observation']
            status = 'OK' if obs.get('success') else 'FAIL'
            print(f'  Obs [{status}]: {str(obs.get("result") or obs.get("error_message", ""))[:120]}')
        print()

In [ ]:
# Print compressed states produced during the episode
if ep_files:
    compressed = raw.get('compressed_states', [])
    print(f'{len(compressed)} compressed state(s)\n')
    for cs in compressed:
        print(f'State at step {cs["step_index"]} (method: {cs.get("compression_method", "?")})')
        print(f'  Decisions made   : {cs.get("decisions_made", [])}')
        print(f'  Open questions   : {cs.get("open_questions", [])}')
        print(f'  Itinerary sketch : {cs.get("current_itinerary_sketch", "")[:200]}')
        print()

## 7. Checkpoint artefacts

After training, checkpoints land in `outputs/checkpoints/`.  
The final checkpoint is in `outputs/checkpoints/final/`.

In [ ]:
from optimized_llm_planning_memory.training.run_manifest import resolve_checkpoint

ckpt_path = resolve_checkpoint(RUN_ID, output_dir=OUTPUT_DIR)
if ckpt_path:
    size_mb = ckpt_path.stat().st_size / 1e6
    print(f'Checkpoint : {ckpt_path}  ({size_mb:.1f} MB)')

    # List what's alongside the zip
    companion_dir = OUTPUT_DIR / 'checkpoints' / f'compressor_step_{NUM_TIMESTEPS}'
    if companion_dir.exists():
        print(f'Companion  : {companion_dir}')
        for f in sorted(companion_dir.rglob('*')):
            print(f'  {f.relative_to(companion_dir)}  ({f.stat().st_size / 1024:.1f} KB)')
else:
    print('No checkpoint found — was checkpoint_every_n_steps <= num_timesteps?')

# Show final/ checkpoint
final_dir = OUTPUT_DIR / 'checkpoints' / 'final'
if final_dir.exists():
    print(f'\nFinal checkpoint: {final_dir}')
    for f in sorted(final_dir.rglob('*')):
        print(f'  {f.relative_to(final_dir)}  ({f.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Verify the checkpoint loads without error
if ckpt_path and ckpt_path.exists():
    from stable_baselines3 import PPO
    try:
        model = PPO.load(ckpt_path, device='cpu')
        print(f'Checkpoint loaded OK')
        print(f'  Policy class : {type(model.policy).__name__}')
        print(f'  Observation space: {model.observation_space}')
        print(f'  Action space     : {model.action_space}')
    except Exception as e:
        print(f'Load failed: {e}')

## 8. Disk usage summary

Useful before a longer Colab run — shows the storage impact of each artifact type.

In [ ]:
from optimized_llm_planning_memory.utils.colab_utils import estimate_run_size

sizes = estimate_run_size(RUN_ID, output_dir=OUTPUT_DIR)
print('Artifact storage breakdown:')
for k, v in sizes.items():
    bar = '█' * max(1, int(v * 5))
    print(f'  {k:<25} {v:>7.2f} MB  {bar}')